# Machine Learning – Day 21
# Silhouette Analysis

This notebook demonstrates how to evaluate clustering quality using **Silhouette Analysis**.

## Objectives
- Generate synthetic clustered data
- Apply K-Means clustering
- Compute the Silhouette Score
- Find the optimal number of clusters (K)
- Visualize Silhouette Scores
- Compare clustering algorithms (K-Means, Hierarchical, DBSCAN)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    calinski_harabasz_score,
    davies_bouldin_score
)

plt.rcParams["figure.figsize"]=(7,5)


## Step 1: Create Sample Dataset

In [ ]:
X, y = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=0.6,
    random_state=42
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

plt.scatter(X_scaled[:,0], X_scaled[:,1], s=30)
plt.title("Synthetic Dataset")
plt.show()


## Step 2: Apply K-Means

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

plt.scatter(X_scaled[:,0], X_scaled[:,1], c=labels, cmap="tab10", s=30)
plt.scatter(kmeans.cluster_centers_[:,0],kmeans.cluster_centers_[:,1],
            color="black",marker="X",s=200,label="Centroids")
plt.legend()
plt.title("K-Means Clusters")
plt.show()


## Step 3: Calculate Silhouette Score

In [ ]:
score = silhouette_score(X_scaled, labels)
sample_scores = silhouette_samples(X_scaled, labels)

print("Overall Silhouette Score:", round(score,4))
print("Minimum Sample Score:", round(sample_scores.min(),4))
print("Maximum Sample Score:", round(sample_scores.max(),4))


## Step 4: Find the Best Value of K

In [ ]:
k_values = range(2,11)
scores=[]
inertia=[]

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    pred = model.fit_predict(X_scaled)

    scores.append(silhouette_score(X_scaled,pred))
    inertia.append(model.inertia_)

best_k = k_values[np.argmax(scores)]
print("Best K:",best_k)


In [ ]:
plt.plot(k_values,scores,marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis")
plt.grid(True)
plt.show()

plt.plot(k_values,inertia,marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True)
plt.show()


## Step 5: Detailed Silhouette Plot

In [ ]:
def silhouette_plot(X, labels):
    n_clusters=len(np.unique(labels))
    values=silhouette_samples(X,labels)
    avg=silhouette_score(X,labels)

    y_lower=10
    plt.figure(figsize=(8,5))

    for i in range(n_clusters):
        vals=values[labels==i]
        vals.sort()
        size=len(vals)
        y_upper=y_lower+size
        plt.fill_betweenx(np.arange(y_lower,y_upper),0,vals,alpha=0.7)
        plt.text(-0.05,y_lower+size/2,str(i))
        y_lower=y_upper+10

    plt.axvline(avg,color="red",linestyle="--",label=f"Average={avg:.3f}")
    plt.xlabel("Silhouette Coefficient")
    plt.ylabel("Clusters")
    plt.title("Silhouette Plot")
    plt.legend()
    plt.show()

silhouette_plot(X_scaled,labels)


## Step 6: Compare Different Clustering Algorithms

In [ ]:
X2,_=make_moons(n_samples=300,noise=0.1,random_state=42)
X2=StandardScaler().fit_transform(X2)

models={
    "K-Means":KMeans(n_clusters=2,random_state=42,n_init=10),
    "Hierarchical":AgglomerativeClustering(n_clusters=2),
    "DBSCAN":DBSCAN(eps=0.3,min_samples=5)
}

for name,model in models.items():
    pred=model.fit_predict(X2)

    mask=pred!=-1
    if len(np.unique(pred[mask]))>1:
        s=silhouette_score(X2[mask],pred[mask])
        print(f"{name}: {s:.4f}")
    else:
        print(name,": Only one cluster found")


## Step 7: Compare Evaluation Metrics

In [ ]:
print("Silhouette Score :",silhouette_score(X_scaled,labels))
print("Calinski-Harabasz :",calinski_harabasz_score(X_scaled,labels))
print("Davies-Bouldin :",davies_bouldin_score(X_scaled,labels))


# Conclusion

- Silhouette Score ranges from **-1 to +1**.
- Higher values indicate compact and well-separated clusters.
- It is useful for selecting the optimal value of **K**.
- Combine Silhouette Analysis with visual inspection and domain knowledge for the best results.
